In [1]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"]="0" 
import torch

# import os
# os.environ["CUDA_VISIBLE_DEVICES"]="1" 
torch.cuda.set_device(0)


import numpy as np
import tensorflow as tf
import pandas as pd
import pyarabic.araby as araby
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.initializers import TruncatedNormal
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.metrics import CategoricalAccuracy
import torch
from sklearn.metrics import accuracy_score, f1_score
from transformers import Trainer, TrainingArguments
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset, Dataset, concatenate_datasets
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', 1000)


log_file = 'log.txt'
with open(log_file, 'w') as f:
    f.write('Model,Accuracy,F1\n')


# df = pd.read_csv('RE_dataset_Binary_Cleaned.csv') #, sep='\t'   , quoting=3 , encoding='utf-8', engine='python', quotechar="'"
# df = pd.read_csv('RE_dataset_Binary_Cleaned.txt', header=None, names=['text'], engine='python', quotechar='"',)


import pandas as pd

file_path = 'RE_dataset_Binary_Cleaned.txt'



with open(file_path, 'r') as file:
    lines = file.readlines()

labels = []
texts = []

for line in lines:
    line = line.strip()
    
    if line[0] == '"':
        if line[1] == 'F':
            label = 'F'
            text = line[3:].strip().lstrip(',')
        elif line[1] == 'N':
            label = 'NF'
            text = line[4:].strip().lstrip(',')
    else:
        if line[0] == 'F':
            label = 'F'
            text = line[2:].strip().lstrip(',')
        elif line[0] == 'N':
            label = 'NF'
            text = line[3:].strip().lstrip(',')
    
    labels.append(label)
    texts.append(text)

df = pd.DataFrame({'label': labels, 'text': texts})





df.fillna('', inplace=True)

display(len(df))
display(df[:4])



df = df[df['text'] != '']

classes = set(df['label'].values)
display(classes)

# return

df['label'] = df['label'].astype('category')
df['label'] = df['label'].cat.codes



df = df[['text', 'label']]


classes_num = len(classes)
display(classes_num)
display(len(df))


ds = Dataset.from_pandas(df)

ds = ds.train_test_split(test_size=0.2, seed=42)
display(ds)

max_sequence_length = 128


models = [ 
        
    
    'microsoft/deberta-v3-base',
    'FacebookAI/roberta-base',
    'bert-base-cased',
]


for model_name in models:
    for i in range(3):
        print(f'{model_name}, try:{i}')
              
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSequenceClassification.from_pretrained(model_name,
                                                              num_labels=classes_num).to('cuda')                                                 
        dataset_train = ds['train']
        dataset_validation = ds['test']                                                    
        
      

        def preprocess_function(examples):
            return tokenizer(examples['text'], truncation=True, padding="max_length",
                            max_length=max_sequence_length, add_special_tokens=True)
        
        
        dataset_train = dataset_train.map(preprocess_function, batched=True)
        dataset_validation = dataset_validation.map(preprocess_function, batched=True)
        
       
        
        def compute_metrics(eval_pred):
            logits, labels = eval_pred
            predictions = np.argmax(logits, axis=-1)    
            acc = accuracy_score(labels, predictions)        
            f1 = f1_score(labels, predictions, average='macro')   
            with open(log_file, 'a') as f:
                f.write(f'{model_name},{acc},{f1}\n')
            return {'accuracy': acc, 'f1_score': f1}

            
        epochs = 25
        save_steps = 10000 #save checkpoint every 10000 steps
        batch_size = 32
        
        training_args = TrainingArguments(
            output_dir = 'bert/',
            overwrite_output_dir=True,
            num_train_epochs = epochs,
            per_device_train_batch_size = batch_size,
            per_device_eval_batch_size = batch_size,
            save_steps = save_steps,
            save_total_limit = 1, #only save the last 5 checkpoints
            fp16=True,
            learning_rate = 5e-5,  # 5e-5 is the default
            logging_steps = 50, #50_000
            evaluation_strategy = 'steps',
            # evaluate_during_training = True,
            eval_steps = 50
            
        )
        
        trainer = Trainer(
            model = model,
            args = training_args,
            # data_collator=data_collator,
            train_dataset=dataset_train,
            eval_dataset=dataset_validation,
            compute_metrics = compute_metrics
        )
        
        
        # trainer.train(resume_from_checkpoint=True)
        trainer.train()


results = pd.read_csv(log_file)

best_results = results.groupby('Model', as_index=False)['F1'].max()

best_results = pd.merge(best_results, results, on=['Model', 'F1'])
best_results = best_results[['Model', 'Accuracy', 'F1']]
best_results = best_results.drop_duplicates()
best_results.to_csv('results_1.csv')
display(best_results)



2024-08-08 14:09:54.382471: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-08-08 14:09:54.405064: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-08-08 14:09:54.800746: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


4976

,label,text
0,NF,"Yoggie shall coordinate on future enhancement and features with our organization.,;"
1,NF,"Within-page links should be clearly distinguishable from other links that lead to a different page. EX. Within-page links are shown with dashed rather than solid underlines,;"
2,F,"With the information provided by administrator user can directly contact with system or he can contact during their chat.,;"
3,F,"""""Whilst on-going, a multi-drivers indication shall be displayed permanently at all Cab radios."""","";"


{'F', 'NF'}

2

...

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/3980 [00:00<?, ? examples/s]

Map:   0%|          | 0/996 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss,Accuracy,F1 Score
50,0.631800,0.484263,0.775100,0.774700
100,0.385600,0.372468,0.835341,0.835299
150,0.303400,0.399199,0.854418,0.853870
200,0.214200,0.601287,0.853414,0.853043
250,0.174600,0.390987,0.863454,0.863440
300,0.129800,0.403708,0.857430,0.856978
350,0.121800,0.407413,0.855422,0.855384
400,0.105800,0.658266,0.853414,0.853043
450,0.101900,0.786691,0.856426,0.856373
500,0.087900,0.593941,0.849398,0.848885


,Model,Accuracy,F1
0,FacebookAI/roberta-base,0.859438,0.859424
3,bert-base-cased,0.863454,0.863440
6,microsoft/deberta-v3-base,0.868474,0.868474


In [ ]:
# without specifying seed
# Model 	Accuracy 	F1
# 0 	FacebookAI/roberta-base 	0.848394 	0.848393
# 2 	bert-base-cased 	        0.839357 	0.839352
# 5 	microsoft/deberta-v3-base 	0.866466 	0.866115